# BÀI THỰC HÀNH 4: MẠNG NEURAL HỒI QUY

## Runtime setup

This notebook keeps the original assignment statements and orchestrates the Python package from GitHub. Set `REPO_URL` below after pushing the repository. Attach or mount small-PhoMT separately; dataset files must not be committed to GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Replace with your GitHub repository URL after pushing the source code.
REPO_URL = "https://github.com/<username>/<repository>.git"
if "<username>" in REPO_URL or "<repository>" in REPO_URL:
    raise ValueError("Set REPO_URL to the real GitHub repository before running.")

if "/kaggle/" in os.getcwd():
    REPO_DIR = Path("/kaggle/working/DS201_lab4")
else:
    REPO_DIR = Path("/content/DS201_lab4")

if not (REPO_DIR / "src" / "lab4").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}")

%pip install -r {REPO_DIR / 'requirements.txt'}
sys.path.insert(0, str(REPO_DIR / "src"))
print("Repository:", REPO_DIR)

### Data path and experiment configuration

Kaggle: attach the small-PhoMT dataset and set `DATA_DIR` to the directory containing the three JSON files. Colab: mount Google Drive and set the same variable to the mounted data directory.

In [ ]:
import os
from pathlib import Path

# Detect the notebook runtime and define its base paths.
if Path("/kaggle/input").is_dir():
    PLATFORM = "kaggle"
    ROOT_DIR = Path("/kaggle/input")
    OUTPUT_ROOT = Path("/kaggle/working/outputs")
elif Path("/content").is_dir():
    PLATFORM = "colab"
    ROOT_DIR = Path("/content")
    OUTPUT_ROOT = ROOT_DIR / "outputs"
else:
    PLATFORM = "local"
    ROOT_DIR = Path.cwd()
    OUTPUT_ROOT = ROOT_DIR / "outputs"

# Set this relative to ROOT_DIR, or override PHOMT_DATA_DIR with an absolute path.
# Examples:
#   Kaggle: "<dataset-name>/small-PhoMT"
#   Colab local upload: "small-PhoMT"
#   Colab Drive: "/content/drive/MyDrive/small-PhoMT"
DATA_RELATIVE_PATH = "<dataset-name>/small-PhoMT" if PLATFORM == "kaggle" else "small-PhoMT"
DATA_DIR = Path(os.environ["PHOMT_DATA_DIR"]) if "PHOMT_DATA_DIR" in os.environ else ROOT_DIR / DATA_RELATIVE_PATH

required_files = ["small-train.json", "small-dev.json", "small-test.json"]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under DATA_DIR={DATA_DIR}. "
        "Set DATA_RELATIVE_PATH or PHOMT_DATA_DIR to the mounted dataset folder."
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from lab4.config import ExperimentConfig
from lab4.experiment import run_experiment

config = ExperimentConfig(
    emb=256, hidden=256, layers=3, dropout=0.3, batch=32,
    lr=1e-3, teacher_ratio=0.5, epochs=10, max_words=50,
    vocab_limit=10_000, min_freq=2, train_limit=None, dev_limit=None, seed=42,
)
print("Platform:", PLATFORM)
print("Root:", ROOT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUTPUT_ROOT)
print("Config:", config)

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses.

Bộ dữ liệu sử dụng: [PhoMT](https://drive.google.com/drive/folders/186OAOuSEYEDVcry7WP5UBdqECXo26QAb?usp=drive_link).

#### Bài 1: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụng độ đo ROUGE-L.

In [ ]:
result_none = run_experiment(
    model_name="none",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_ROOT / "assignment_1_no_attention",
    config=config,
    evaluate_test=True,
)
print(result_none)

#### Bài 2: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [ ]:
result_bahdanau = run_experiment(
    model_name="bahdanau",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_ROOT / "assignment_2_bahdanau",
    config=config,
    evaluate_test=True,
)
print(result_bahdanau)

#### Bài 3: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [ ]:
result_luong = run_experiment(
    model_name="luong",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_ROOT / "assignment_3_luong",
    config=config,
    evaluate_test=True,
)
print(result_luong)

## Results summary

Collect the results saved by the three assignments into a comparison table. Detailed CSV files and plots remain in each assignment output directory.

In [ ]:
import pandas as pd

results = [result_none, result_bahdanau, result_luong]
comparison = pd.DataFrame([
    {
        "model": r.name,
        "best_epoch": r.best_epoch,
        "dev_rougeL_f1": r.dev_metrics["rougeL_f1"],
        "test_rougeL_f1": r.test_metrics["rougeL_f1"],
        "parameter_count": r.parameter_count,
        "seconds_per_epoch": r.seconds_per_epoch,
    }
    for r in results
])
comparison.to_csv(OUTPUT_ROOT / "comparison.csv", index=False)
display(comparison)
print("Comparison saved to:", OUTPUT_ROOT / "comparison.csv")

## Download run artifacts

Generated files stay in the runtime output directory and are not added to the cloned repository. Create a ZIP for download when needed; in Colab, the next cell starts a browser download. In Kaggle, download the ZIP from the notebook output files.

In [ ]:
import shutil

ARTIFACTS_ZIP = shutil.make_archive(
    str(OUTPUT_ROOT.parent / "lab4_outputs"),
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Artifacts archive:", ARTIFACTS_ZIP)

if PLATFORM == "colab":
    from google.colab import files
    files.download(ARTIFACTS_ZIP)